# Neural network basics

A neural network is a stack of simple units. Each unit takes a weighted sum of its inputs and bends it with a non-linear function. One unit is a perceptron. Many units in layers make a multilayer perceptron (MLP). This notebook builds both in NumPy, then does the same job in Keras 3 on MNIST digits and California house prices.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
from common.dl import get_device, set_seed, setup_keras

setup_keras("torch")
import keras
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from common.datasets import load_california_housing, load_mnist, make_moons_data
from common.plotting import plot_decision_boundary, plot_history

SEED = 42
set_seed(SEED)
keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)
print("device:", get_device())

## 1. Intuition

A unit asks one question: is the weighted evidence strong enough? Inputs are multiplied by weights, added to a bias, and passed through an **activation**. A perceptron uses a hard step. It draws one straight line and says "above" or "below".

One line cannot solve every problem. XOR is the classic case: no single line separates (0,0) and (1,1) from (0,1) and (1,0). A second layer of units fixes it by bending the space first, then drawing a line. That is all deep learning adds: more layers of the same unit.

## 2. Maths and cost function

One unit: $a=\sigma(\mathbf w^\top\mathbf x+b)$. A layer stacks units, so $\mathbf a=\sigma(W\mathbf x+\mathbf b)$. An MLP chains layers:

$$\hat y=\sigma_2\big(W_2\,\sigma_1(W_1\mathbf x+\mathbf b_1)+\mathbf b_2\big)$$

The cost function depends on the task:

- Regression: mean squared error, $J=\frac1n\sum(\hat y-y)^2$.
- Binary classification: cross-entropy, $J=-\frac1n\sum\,[y\log\hat y+(1-y)\log(1-\hat y)]$, with a sigmoid output.
- Many classes: the same idea with a softmax output.

Without the non-linearity $\sigma_1$, $W_2W_1$ collapses into one matrix and the network is a linear model. The activation is what buys depth.

## 3. From scratch

First the perceptron on XOR. The classic update nudges the weights by the error on each point. It cannot converge on XOR, so we count the mistakes.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])


def train_perceptron(X: np.ndarray, y: np.ndarray, epochs: int = 50, lr: float = 0.1):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epochs):
        for xi, yi in zip(X, y):
            err = yi - int(xi @ w + b > 0)  # +1, 0 or -1
            w, b = w + lr * err * xi, b + lr * err
    return w, b


w, b = train_perceptron(X_xor, y_xor)
pred = (X_xor @ w + b > 0).astype(int)
print("perceptron predictions:", pred, "truth:", y_xor, "wrong:", (pred != y_xor).sum())

The perceptron always gets at least one of four wrong. Now an MLP with one hidden layer of tanh units, trained by gradient descent on the moons data. The gradients are written out by hand here and derived properly in notebook 02.

In [ ]:
Xm_df, ym = make_moons_data(400, noise=0.2, seed=SEED)
Xm, ym = Xm_df.to_numpy(), np.asarray(ym)


class NumpyMLP:
    def __init__(self, n_in: int, n_hidden: int, seed: int = 0) -> None:
        g = np.random.default_rng(seed)
        self.W1 = g.normal(0, 1 / np.sqrt(n_in), (n_in, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.W2 = g.normal(0, 1 / np.sqrt(n_hidden), (n_hidden, 1))
        self.b2 = np.zeros(1)

    def forward(self, X: np.ndarray):
        h = np.tanh(X @ self.W1 + self.b1)
        p = 1 / (1 + np.exp(-(h @ self.W2 + self.b2)))
        return h, p.ravel()

    def fit(self, X: np.ndarray, y: np.ndarray, epochs: int = 2000, lr: float = 0.5) -> list[float]:
        losses = []
        for _ in range(epochs):
            h, p = self.forward(X)
            losses.append(-np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9)))
            d_out = (p - y)[:, None] / len(X)  # cross-entropy with sigmoid
            d_h = (d_out @ self.W2.T) * (1 - h**2)  # tanh derivative
            self.W2 -= lr * h.T @ d_out
            self.b2 -= lr * d_out.sum(axis=0)
            self.W1 -= lr * X.T @ d_h
            self.b1 -= lr * d_h.sum(axis=0)
        return losses

    def predict(self, X: np.ndarray) -> np.ndarray:
        return (self.forward(X)[1] > 0.5).astype(int)


net = NumpyMLP(2, 8, seed=SEED)
losses = net.fit(Xm, ym)
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}, accuracy {(net.predict(Xm) == ym).mean():.3f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(losses)
ax[0].set_title("training loss")
plot_decision_boundary(net, Xm, ym, ax[1])
plt.show()

Loss falls from about 0.7 (a coin flip) to under 0.1, and the boundary bends around the moons, which no linear model can do. Eight hidden units are enough.

## 4. Library version

Keras does the same with three lines of architecture. First classification: MNIST digits, flattened to 784 numbers, a 128-unit hidden layer and a 10-way softmax. We use 5,000 training images to stay fast.

In [ ]:
x_tr, y_tr, x_te, y_te = load_mnist(5000, 1000, seed=SEED)
x_tr, x_te = x_tr.reshape(len(x_tr), -1), x_te.reshape(len(x_te), -1)

set_seed(SEED)
clf = keras.Sequential(
    [
        keras.Input((784,)),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ]
)
clf.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
clf.summary()
hist = clf.fit(x_tr, y_tr, epochs=10, batch_size=128, validation_split=0.2, verbose=0)
print("test accuracy:", round(clf.evaluate(x_te, y_te, verbose=0)[1], 3))
plot_history(hist.history, ("loss", "val_loss"), title="MNIST loss")
plt.show()

About 91% on 1,000 unseen digits from 5,000 training images, in a few seconds. The validation loss flattens while the training loss keeps falling, an early sign of overfitting that notebook 06 deals with.

Now regression. California housing predicts a price from eight numbers. The output layer has one unit and no activation, and the loss is MSE. Inputs are standardised first, using training statistics only.

In [ ]:
Xh, yh = load_california_housing(6000, seed=SEED)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh.to_numpy(), yh.to_numpy(), random_state=SEED)
scaler = StandardScaler().fit(Xh_tr)


def make_reg(width: int = 32) -> keras.Model:
    set_seed(SEED)
    model = keras.Sequential(
        [
            keras.Input((8,)),
            keras.layers.Dense(width, activation="relu"),
            keras.layers.Dense(width, activation="relu"),
            keras.layers.Dense(1),
        ]
    )
    model.compile(optimizer="adam", loss="mse")
    return model


reg = make_reg()
h = reg.fit(
    scaler.transform(Xh_tr), yh_tr, epochs=40, batch_size=256, validation_split=0.2, verbose=0
)
mse = reg.evaluate(scaler.transform(Xh_te), yh_te, verbose=0)
print(f"test MSE {mse:.3f}, baseline (predict the mean) {yh_te.var():.3f}")

## 5. Tuning

Width is the first knob. More units fit more detail, at the cost of time and overfitting risk.

In [ ]:
for width in (4, 16, 64, 256):
    model = make_reg(width)
    model.fit(scaler.transform(Xh_tr), yh_tr, epochs=40, batch_size=256, verbose=0)
    print(
        f"width {width:>3}: params {model.count_params():>6}, test MSE {model.evaluate(scaler.transform(Xh_te), yh_te, verbose=0):.3f}"
    )

Test MSE falls from 1.07 at width 4 (barely better than the mean) to 0.34 at width 64, then only to 0.31 at width 256 with 14 times the parameters. Returns shrink fast. Depth, activation and learning rate are tuned the same way in later notebooks.

## 6. What breaks

Skip the scaling. Median income is about 4 and population is about 1,400, so the large inputs dominate the weighted sums and the first-layer gradients.

In [ ]:
raw = make_reg()
raw.fit(Xh_tr, yh_tr, epochs=40, batch_size=256, verbose=0)
print(f"raw features: test MSE {raw.evaluate(Xh_te, yh_te, verbose=0):.3f}")

The raw-feature network scores about 2.5 times worse than the scaled one (1.0 against 0.39), close to predicting the mean at 1.38. Same model, same epochs, only the input scale changed. Always standardise inputs for a neural network.